<!-- type: tutorial -->
# Triage-Driven vs Naive Forecasting on M4 Monthly (v0.4.0)

This walkthrough compares two forecasting strategies on 100 M4 monthly series:

| Strategy | Description |
|---|---|
| **Triage-driven** | Run forecastability triage → `ForecastPrepContract` → choose a `statsforecast` model family from `contract.recommended_families` |
| **SeasonalNaive baseline** | `SeasonalNaive(season_length=12)` — the standard M4 competition benchmark |

> **Scope.** Triage and contract building use only the public `forecastability` facade.
> `statsforecast` model fitting is the downstream consumer; the `forecastability`
> package does not depend on it.

The output CSV `outputs/m4_routing_comparison.csv` records per-series results.


In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from forecastability import (
    ForecastPrepContract,
    TriageRequest,
    build_forecast_prep_contract,
    run_triage,
)
from forecastability.triage import AnalysisGoal

SEED = 42
N_SERIES = 100       # limit to 100 series for runtime budget
M4_HORIZON = 18      # M4 monthly competition horizon
MIN_TRAIN = 48       # minimum training observations required
MAX_LAG = 24
N_SURROGATES = 99

DATA_DIR = Path("data/m4")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = Path("outputs/notebooks/walkthroughs/06_triage_driven_vs_naive_on_m4")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = Path("outputs/m4_routing_comparison.csv")


## 1. Download M4 monthly data

In [2]:
M4_URL = (
    "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/"
    "Dataset/Train/Monthly-train.csv"
)
M4_RAW = DATA_DIR / "Monthly-train.csv"

if not M4_RAW.exists():
    print("Downloading M4 monthly training data ...")
    urlretrieve(M4_URL, M4_RAW)  # noqa: S310 — trusted, hardcoded URL
    print(f"Downloaded to {M4_RAW}")
else:
    print(f"Using cached {M4_RAW}")

# Load and convert to long format
wide_df = pd.read_csv(M4_RAW)
id_col = str(wide_df.columns[0])
long_df = (
    wide_df.rename(columns={id_col: "unique_id"})
    .melt(id_vars="unique_id", value_name="y")
    .dropna(subset=["y"])
)
long_df["unique_id"] = long_df["unique_id"].astype(str)
long_df["timestamp"] = long_df.groupby("unique_id").cumcount()
long_df = long_df[["unique_id", "timestamp", "y"]].sort_values(["unique_id", "timestamp"])

n_total = long_df["unique_id"].nunique()
print(f"Total M4 monthly series: {n_total}")
print(long_df.head())


Downloaded to data/m4/Monthly-train.csv
Total M4 monthly series: 48000
       unique_id  timestamp       y
0             M1          0  8000.0
48000         M1          1  8350.0
96000         M1          2  8570.0
144000        M1          3  7700.0
192000        M1          4  7080.0


## 2. Sample series and run triage

In [ ]:
# Filter series with sufficient training observations
series_lengths = long_df.groupby("unique_id")["y"].count()
eligible = series_lengths[
    series_lengths >= MIN_TRAIN + M4_HORIZON
].index.tolist()
print(f"Eligible series (>= {MIN_TRAIN + M4_HORIZON} obs): {len(eligible)}")

# Sample N_SERIES deterministically
rng = np.random.default_rng(SEED)
sampled_ids = sorted(
    rng.choice(eligible, size=min(N_SERIES, len(eligible)), replace=False)
)
print(f"Selected {len(sampled_ids)} series")

# Run triage per series
triage_results: list[dict] = []
for i, sid in enumerate(sampled_ids):
    vals = long_df[long_df["unique_id"] == sid]["y"].to_numpy(dtype=float)
    train_vals = vals[: -M4_HORIZON]
    test_vals = vals[-M4_HORIZON :]

    triage = run_triage(
        TriageRequest(
            series=train_vals,
            goal=AnalysisGoal.univariate,
            max_lag=MAX_LAG,
            n_surrogates=N_SURROGATES,
            random_state=SEED,
        )
    )
    contract = build_forecast_prep_contract(
        triage,
        horizon=M4_HORIZON,
        target_frequency="MS",
        add_calendar_features=False,
    )
    triage_results.append({
        "series_id": sid,
        "n_train": len(train_vals),
        "n_test": len(test_vals),
        "blocked": contract.blocked,
        "forecastability_class": contract.forecastability_class,
        "recommended_families": contract.recommended_families,
        "recommended_target_lags": contract.recommended_target_lags,
        "_train": train_vals,
        "_test": test_vals,
    })
    if (i + 1) % 10 == 0:
        print(f"  Triaged {i + 1}/{len(sampled_ids)} series ...")

print(
    f"Triage complete. "
    f"{sum(1 for r in triage_results if r['blocked'])} series blocked."
)


Eligible series (>= 66 obs): 47607
Selected 100 series
  Triaged 10/100 series ...
  Triaged 20/100 series ...
  Triaged 30/100 series ...


## 3. Fit triage-driven and baseline models

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, ETS, SeasonalNaive

SEASON_LENGTH = 12  # monthly data → 12-month seasonality


def _pick_model(recommended_families: list[str]) -> object:
    """Map recommended_families to a statsforecast model."""
    if "arima" in recommended_families:
        return AutoARIMA(season_length=SEASON_LENGTH)
    if any(
        f in recommended_families
        for f in ("ets", "linear_state_space", "seasonal_state_space")
    ):
        return ETS(season_length=SEASON_LENGTH)
    return SeasonalNaive(season_length=SEASON_LENGTH)


def _smape(actual: np.ndarray, predicted: np.ndarray) -> float:
    denom = (np.abs(actual) + np.abs(predicted)) / 2
    denom = np.where(denom == 0, 1e-8, denom)
    return float(np.mean(np.abs(actual - predicted) / denom) * 100)


def _fit_predict_sf(model: object, train: np.ndarray, horizon: int) -> np.ndarray:
    """Fit a statsforecast model on train and predict horizon steps."""
    df = pd.DataFrame({
        "unique_id": "s",
        "ds": pd.date_range("2000-01", periods=len(train), freq="MS"),
        "y": train,
    })
    sf = StatsForecast(models=[model], freq="MS")
    sf.fit(df)
    fc = sf.predict(horizon)
    return fc.iloc[:, -1].to_numpy()


comparison_rows: list[dict] = []
for i, row in enumerate(triage_results):
    train = row["_train"]
    test = row["_test"]
    sid = row["series_id"]
    rec_families = row["recommended_families"]

    triage_model = _pick_model(rec_families)
    triage_model_name = type(triage_model).__name__
    naive_model = SeasonalNaive(season_length=SEASON_LENGTH)

    try:
        triage_preds = _fit_predict_sf(triage_model, train, M4_HORIZON)
        naive_preds = _fit_predict_sf(naive_model, train, M4_HORIZON)
        smape_triage = _smape(test, triage_preds)
        smape_naive = _smape(test, naive_preds)
        triage_wins = bool(smape_triage < smape_naive)
        error = None
    except Exception as exc:
        smape_triage = float("nan")
        smape_naive = float("nan")
        triage_wins = False
        error = str(exc)

    comparison_rows.append({
        "series_id": sid,
        "n_train": row["n_train"],
        "blocked": row["blocked"],
        "forecastability_class": row["forecastability_class"],
        "recommended_families": "|".join(rec_families) if rec_families else "none",
        "triage_model": triage_model_name,
        "smape_triage": smape_triage,
        "smape_naive": smape_naive,
        "triage_wins": triage_wins,
        "error": error,
    })

    if (i + 1) % 10 == 0:
        print(f"  Evaluated {i + 1}/{len(triage_results)} series ...")

comp_df = pd.DataFrame(comparison_rows)
print(
    f"\nComplete. {comp_df['triage_wins'].sum()}/{len(comp_df)} "
    "series: triage-driven wins."
)


## 4. Aggregate results and comparison table

In [ ]:
valid = comp_df.dropna(subset=["smape_triage", "smape_naive"])
print(f"Valid comparisons: {len(valid)}/{len(comp_df)}")
print()

# Aggregate by model type chosen
agg = (
    valid.groupby("triage_model")[["smape_triage", "smape_naive"]]
    .agg(["mean", "median", "count"])
)
print("sMAPE by triage-chosen model:")
display(agg)

# Overall summary
overall = {
    "mean sMAPE (triage-driven)": f"{valid['smape_triage'].mean():.2f}%",
    "mean sMAPE (SeasonalNaive)": f"{valid['smape_naive'].mean():.2f}%",
    "median sMAPE (triage-driven)": f"{valid['smape_triage'].median():.2f}%",
    "median sMAPE (SeasonalNaive)": f"{valid['smape_naive'].median():.2f}%",
    "triage wins": (
        f"{valid['triage_wins'].sum()}/{len(valid)} "
        f"({valid['triage_wins'].mean():.1%})"
    ),
}

display(Markdown("### Overall summary\n"))
display(Markdown("\n".join(f"- **{k}:** {v}" for k, v in overall.items())))


## 5. Plot and save results

In [ ]:
# sMAPE scatter: triage-driven vs SeasonalNaive
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(
    valid["smape_naive"], valid["smape_triage"],
    alpha=0.4, s=20, color="#2196F3"
)
lim = max(valid["smape_naive"].max(), valid["smape_triage"].max()) * 1.05
axes[0].plot([0, lim], [0, lim], "k--", linewidth=0.8)
axes[0].set_xlabel("sMAPE — SeasonalNaive (%)")
axes[0].set_ylabel("sMAPE — Triage-driven (%)")
axes[0].set_title(
    "Triage-driven vs Seasonal Naive\n"
    "(points below diagonal = triage wins)"
)

# Distribution boxplot
axes[1].boxplot(
    [valid["smape_triage"].dropna(), valid["smape_naive"].dropna()],
    labels=["Triage-driven", "SeasonalNaive"],
    patch_artist=True,
    boxprops=dict(facecolor="#90CAF9"),
)
axes[1].set_ylabel("sMAPE (%)")
axes[1].set_title("sMAPE distribution")

plt.suptitle(f"M4 Monthly — {len(valid)} series", fontsize=12)
plt.tight_layout()

fig_path = OUTPUT_ROOT / "smape_comparison.png"
plt.savefig(fig_path, dpi=100)
plt.show()
print(f"Figure saved to {fig_path}")

# Save CSV
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
comp_df.to_csv(OUTPUT_CSV, index=False)
print(f"Comparison CSV saved to {OUTPUT_CSV}")


## Summary

This walkthrough ran forecastability triage on 100 M4 monthly series and compared
triage-driven model selection against `SeasonalNaive(season_length=12)`:

- **Triage-driven path:** `run_triage` → `build_forecast_prep_contract` → pick a
  `statsforecast` model from `contract.recommended_families`.
- **Baseline:** `SeasonalNaive(season_length=12)`.

The `ForecastPrepContract.recommended_families` field drove model selection in a
deterministic, framework-agnostic way. The downstream fitting and prediction used
only `statsforecast` — no framework-specific logic lives in the `forecastability`
package.

**Where to go next:**
- `walkthroughs/05_forecast_prep_to_models.ipynb` — single-series deep-dive into
  the contract wiring pattern.
- `docs/recipes/forecast_prep_to_external_frameworks.md` in the core repo — text
  recipe for Darts, MLForecast, and StatsForecast wiring.
